# 2. Emulator comparison

The model is too expensive to evaluate at every step of an MCMC, so it is replaced by an
emulator: a Gaussian process (GP) that is trained on the model output at the design points and
predicts the observables with an uncertainty at any parameter point. This notebook trains all
emulators of `gpbayestools` on the first 500 design points of notebook 1 and validates them on the
remaining 500 points:

| Emulator | Class | Method |
|---|---|---|
| Scikit-learn | `EmulatorSklearn` | GP of scikit-learn for each principal component (PC) of the observables |
| PCGP | `EmulatorBAND(method="PCGP")` | PC Gaussian process of surmise (BAND collaboration) |
| PCSK | `EmulatorBAND(method="PCSK")` | PC stochastic kriging of surmise, uses the uncertainties of the training data |
| HetGP | `EmulatorHetGP` | heteroscedastic GP of hetGPy for each PC |
| SparseGP | `EmulatorSparseGP` | sparse variational GP ensemble (JAX), for large designs |

Each emulator is trained twice, on the observables and on their logarithm (`log_trafo=True`).
The trained emulators are saved for the closure tests of notebook 3.

In [ ]:
import logging
import time
import warnings
from pathlib import Path

import dill
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

from gpbayestools.emulator_band import EmulatorBAND
from gpbayestools.emulator_hetgp import EmulatorHetGP
from gpbayestools.emulator_sklearn import EmulatorSklearn
from gpbayestools.emulator_sparse_gp import EmulatorSparseGP

logging.basicConfig(level=logging.WARNING)
# convergence warnings of the GP optimizers
warnings.filterwarnings("ignore")

parameter_file = "data/parameters_DIS.txt"
training_file = "data/training_data_DIS.pkl"
figure_dir = Path("figures")
emulator_dir = Path("output/emulators")
figure_dir.mkdir(exist_ok=True)
emulator_dir.mkdir(parents=True, exist_ok=True)

n_train = 500

## Emulator settings

The settings are reasonable choices for this data set, not the result of a tuning:

- The emulators with a PCA keep the PCs that explain 99.99% of the variance of the standardized
  observables (`npc=0.9999`). surmise chooses the PCs itself (see the `EmulatorBAND` docstring).
- `EmulatorSklearn` restarts the optimization of the kernel hyperparameters twice
  (`n_restarts=2`).
- `EmulatorSparseGP` uses a single sparse GP with 250 inducing points and early stopping. A
  bootstrap ensemble (`n_ensemble=5, bootstrap=True`) with 150 inducing points per member was
  less accurate here and five times slower. With 500 training points a sparse GP is not needed,
  it is included for the comparison.
- The simulations are noise-free and the 6.59% is an assigned model uncertainty, so all
  emulators get `errors_are_noise=False`. Otherwise the emulators with a PCA would remove this
  "noise" from the uncertainty of the discarded PCs (HetGP then underestimates its uncertainty
  by a factor of about 3), and the sparse GP would treat it as observation noise. PCSK always
  models the uncertainties as noise of the simulations, which is how it was used in the paper.
- The seeds make the trainings reproducible.

All emulators are trained with `train_emulator(mask)` on the training points selected by the
boolean `mask`. The other points are not used in the training.

In [ ]:
EMULATORS = {
    "Scikit-learn": (
        lambda log: EmulatorSklearn(
            training_file, parameter_file, npc=0.9999, n_restarts=2, seed=1, log_trafo=log,
            errors_are_noise=False,
        ),
        {},
    ),
    "PCGP": (
        lambda log: EmulatorBAND(
            training_file, parameter_file, method="PCGP", seed=1, log_trafo=log
        ),
        {},
    ),
    "PCSK": (
        lambda log: EmulatorBAND(
            training_file, parameter_file, method="PCSK", seed=1, log_trafo=log
        ),
        {},
    ),
    "HetGP": (
        lambda log: EmulatorHetGP(
            training_file, parameter_file, npc=0.9999, log_trafo=log, errors_are_noise=False
        ),
        {},
    ),
    "SparseGP": (
        lambda log: EmulatorSparseGP(
            training_file,
            parameter_file,
            npc=0.9999,
            n_inducing=250,
            seed=1,
            log_trafo=log,
            errors_are_noise=False,
        ),
        # arguments of the training (see PCASparseGPEmulator.fit)
        {"steps": 5000, "early_stopping": True, "patience": 50, "verbose": False},
    ),
}

## Validation metrics

At the $N = 500$ validation points with model values $y_{ij}$ (point $i$, observable $j$), the
emulator predicts the mean $\mu_{ij}$ and the standard deviation $\sigma_{ij}$. For each
observable, the two metrics of
[H. Roch, S. A. Jahan and C. Shen, arXiv:2405.12019](https://arxiv.org/abs/2405.12019)
(Eqs. (10) and (11)) are computed:

**RMS relative error** of the prediction, in the original scale of the observables:

$$\mathcal{E}_j = \sqrt{\frac{1}{N}\sum_i \left(\frac{\mu_{ij} - y_{ij}}{y_{ij}}\right)^2}.$$

**Honesty** of the predicted uncertainty, the logarithm of the RMS of the residuals in units of
the predicted uncertainty:

$$\mathcal{H}_j = \ln \sqrt{\frac{1}{N}\sum_i \frac{(\mu_{ij} - y_{ij})^2}{\sigma_{ij}^2}}.$$

$\mathcal{H} \approx 0$ means that the uncertainty is calibrated, $\mathcal{H} > 0$ that the
emulator is overconfident and $\mathcal{H} < 0$ that it is conservative; with the logarithm,
an uncertainty that is too small or too large by the same factor gives the same $|\mathcal{H}|$.
The honesty is computed
in the space in which the emulator predicts a Gaussian, i.e. for `log_trafo=True` in log space.
Two variants are shown:

- **Honesty with the data uncertainty**, the definition of arXiv:2405.12019:
  $\sigma_{ij}^2 = \sigma_{\mathrm{emu},ij}^2 + \sigma_{\mathrm{data},ij}^2$ combines the
  emulator uncertainty with the uncertainty of the validation data, here the 6.59%. This is also
  the combination that enters the likelihood of the Bayesian analysis. For simulations with
  statistical noise it is the calibration test. Here the model values are noise-free, so it is
  strongly negative and shows how small the emulator uncertainty is compared with the 6.59%.
- **Honesty of the emulator** alone: $\sigma_{ij}$ is only the uncertainty of the emulated
  model function returned by `predict()`. Since the validation values are noise-free, this tests
  the emulator uncertainty itself.

For noisy simulations, `predict(..., include_noise=True)` would give the uncertainty of a new
noisy simulation instead, which the validation functions `test_emulator_errors` of the emulators
use.

In [ ]:
def rms_relative_error(mean, values, log):
    # relative error in the original scale of the observables
    rel = np.expm1(mean - values) if log else (mean - values) / values
    return np.sqrt(np.mean(rel**2, axis=0))


def honesty(mean, std, values, data_err=None):
    var = std**2 if data_err is None else std**2 + data_err**2
    return np.log(np.sqrt(np.mean((mean - values) ** 2 / var, axis=0)))

## Training and validation

This takes about 25 minutes on a laptop, most of it for HetGP and SparseGP.

In [ ]:
results = {}
for log in (False, True):
    for name, (make, train_kwargs) in EMULATORS.items():
        emu = make(log)
        mask = np.zeros(emu.n_ev, dtype=bool)
        mask[:n_train] = True

        start = time.perf_counter()
        emu.train_emulator(mask, **train_kwargs)
        train_time = time.perf_counter() - start

        # the validation points; for log_trafo=True, model_data and model_data_err are
        # the log values and the relative errors
        start = time.perf_counter()
        mean, cov = emu.predict(emu.design_points[~mask])
        predict_time = time.perf_counter() - start
        std = np.sqrt(np.diagonal(cov, axis1=1, axis2=2))
        values = emu.model_data[~mask]
        data_err = emu.model_data_err[~mask]

        results[name, log] = {
            "rms": rms_relative_error(mean, values, log),
            "honesty": honesty(mean, std, values),
            "honesty_data": honesty(mean, std, values, data_err),
            "train_time": train_time,
            "predict_time": predict_time,
            # predictions and validation data, in log space for log_trafo=True
            "mean": mean,
            "std": std,
            "values": values,
            "data_err": data_err,
        }
        with open(emulator_dir / f"{name}_{'log' if log else 'lin'}.dill", "wb") as f:
            dill.dump(emu, f)
        print(f"{name:12s} log_trafo={log!s:5s} trained in {train_time:6.1f} s")

In [ ]:
header = (
    f"{'emulator':12s} {'log':>5s} {'mean E':>8s} {'max E':>8s} {'mean H':>8s} "
    f"{'mean H (data)':>14s} {'train [s]':>10s} {'predict [s]':>12s}"
)
print(header)
print("-" * len(header))
for (name, log), r in results.items():
    print(
        f"{name:12s} {log!s:>5s} {r['rms'].mean():8.4f} {r['rms'].max():8.4f} "
        f"{r['honesty'].mean():8.2f} {r['honesty_data'].mean():14.2f} "
        f"{r['train_time']:10.1f} {r['predict_time']:12.2f}"
    )

The prediction time is for all 500 validation points at once, including the covariance matrices.

In [ ]:
markers = {"Scikit-learn": "o", "PCGP": "s", "PCSK": "D", "HetGP": "^", "SparseGP": "v"}


def plot_metric(key, ylabel, filename, log_scale):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for ax, log in zip(axes, (False, True)):
        for name in EMULATORS:
            ax.plot(results[name, log][key], markers[name], ms=3.5, label=name)
        if log_scale:
            ax.set_yscale("log")
            ax.yaxis.set_major_locator(ticker.LogLocator(subs=[1.0, 2.0, 5.0]))
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:g}"))
            ax.yaxis.set_minor_formatter(ticker.NullFormatter())
        else:
            ax.axhline(0, color="k", ls=":", zorder=-1)
        ax.set_xlabel("observable index")
        ax.set_title("log_trafo=True" if log else "log_trafo=False")
    if not log_scale:
        # symmetric around 0
        ymax = 1.1 * max(abs(y) for ax in axes for y in ax.get_ylim())
        axes[0].set_ylim(-ymax, ymax)
    axes[0].set_ylabel(ylabel)
    axes[1].legend(loc="upper right", ncols=2, fontsize=9)
    fig.tight_layout()
    fig.savefig(figure_dir / filename, dpi=150)


plot_metric("rms", r"RMS relative error $\mathcal{E}$", "02_rms_error.png", log_scale=True)
plot_metric("honesty", r"honesty $\mathcal{H}$", "02_honesty.png", log_scale=False)
plot_metric(
    "honesty_data",
    r"honesty $\mathcal{H}$ with data uncertainty",
    "02_honesty_with_data_uncertainty.png",
    log_scale=False,
)

## Predictions against the true values

For a few observables (rows), the predictions of each emulator trained on the log of the
observables (columns) are plotted against the model values at the 500 validation points. The
error bars are the emulator uncertainty (vertical) and the 6.59% uncertainty of the model values
(horizontal). A perfect emulator lies on the diagonal. The emulators are accurate to better
than 1%, so the row below each panel shows the relative residuals, prediction / model value − 1,
with the emulator uncertainty; the grey band is the 6.59% uncertainty of the model values. The log-space predictions are transformed
back to the original scale, $\exp(\mu)$ with the uncertainty $\exp(\mu)\,\sigma$.

In [ ]:
observables = [0, 33, 66, 99]

n_rows = 2 * len(observables)
fig, axes = plt.subplots(
    n_rows,
    len(EMULATORS),
    figsize=(2.7 * len(EMULATORS), 3.6 * len(observables)),
    height_ratios=[2, 1] * len(observables),
    sharex="col",
)
for col, name in enumerate(EMULATORS):
    r = results[name, True]
    pred = np.exp(r["mean"])
    pred_err = pred * r["std"]
    true = np.exp(r["values"])
    true_err = true * r["data_err"]
    for k, j in enumerate(observables):
        ax, ax_res = axes[2 * k, col], axes[2 * k + 1, col]
        ax.errorbar(
            true[:, j], pred[:, j], xerr=true_err[:, j], yerr=pred_err[:, j],
            fmt=".", ms=2, lw=0.5, alpha=0.5, color=f"C{col}",
        )
        lims = [true[:, j].min(), true[:, j].max()]
        ax.plot(lims, lims, "k-", lw=0.8)
        ax.set_yscale("log")
        # relative residuals with the emulator uncertainty, and the 6.59% for scale
        ax_res.fill_between(lims, -r["data_err"][0, j], r["data_err"][0, j], color="0.9")
        ax_res.errorbar(
            true[:, j], pred[:, j] / true[:, j] - 1, yerr=pred_err[:, j] / true[:, j],
            fmt=".", ms=2, lw=0.5, alpha=0.5, color=f"C{col}",
        )
        ax_res.axhline(0, color="k", lw=0.8)
        ax_res.set_ylim(-0.08, 0.08)
        if col == 0:
            ax.set_ylabel(f"observable {j}\nprediction")
            ax_res.set_ylabel("pred./true - 1")
        else:
            ax_res.set_yticklabels([])
        if k == 0:
            ax.set_title(name)
    axes[-1, col].set_xscale("log")
    axes[-1, col].set_xlabel("model value")
fig.tight_layout()
fig.savefig(figure_dir / "02_prediction_vs_truth.png", dpi=150)

## Discussion

The numbers below are from a run on a laptop (CPU); they can differ slightly between machines.

- **Log transformation.** Training on the log of the observables reduces the error of all
  emulators by a factor of 2–3, since the observables span more than two orders of magnitude.
  With `log_trafo=True`, Scikit-learn, PCGP, PCSK and HetGP reach a mean RMS relative error of
  about 0.5%, far below the 6.59% model uncertainty. The following notebook uses these emulators.
- **SparseGP** is less accurate (about 1%) and conservative ($\mathcal{H} \approx -1.5$, i.e. its
  uncertainty is about $e^{1.5} \approx 4.5$ times too large). This is expected for a sparse variational GP, whose
  uncertainty is overestimated when there are fewer inducing points than training points. It is
  meant for designs with thousands of points, for which the other emulators become slow.
- **Calibration.** HetGP is the best calibrated emulator here ($\mathcal{H} \approx 0.1$), PCGP is
  slightly overconfident ($\mathcal{H} \approx 0.3$) and Scikit-learn conservative
  ($\mathcal{H} \approx -0.9$). PCSK is overconfident ($\mathcal{H} \approx 1.2$, a factor of about 3):
  it models the 6.59%
  as noise of the simulations, so the uncertainty of the emulated function is too small for
  noise-free simulations. `predict(..., include_noise=True)` adds the 6.59% noise to it.
- **Relevance for the Bayesian analysis.** With the data uncertainty, $\mathcal{H}$ is between
  about $-2.7$ and $-1.8$ for all emulators: the emulator errors are much smaller than the 6.59%
  uncertainty of the data. The emulator therefore has only a small effect on the posterior,
  which the closure tests in the [next notebook](03_closure_tests.ipynb) show.